# Comparando Modelos

Estamos en la  Clase 02
<br> El objetivo de la materia es lograr la mejor predicción para nuestro problema, para lo que en próximas clases se probarán pipelines con decenas de alternativas, será indispensable comparar varios modelos predictivos entre sí y decidir cual es el mejor
<br> La tarea no es tan sencilla

## Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


<br>los siguientes comandos estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/labo1"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo1" /content/buckets/b1


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"

--2026-09-22 22:19:09--  https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/dataset_pequeno.csv
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.206.207, 142.250.152.207, 74.125.126.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.206.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 167807380 (160M) [text/csv]
Saving to: ‘/content/buckets/b1/datasets/dataset_pequeno.csv’

/content/buckets/b1 100%[===================>] 160.03M  52.0MB/s    in 3.1s    

2026-09-22 22:19:12 (52.0 MB/s) - ‘/content/buckets/b1/datasets/dataset_pequeno.csv’ saved [167807380/167807380]



# 1  Particion Training/Testing
## Clase 02  Experimento 1

## 1.1  Objetivos
Dado que es el primer experimento de la asignatura, acercar a l@s estudiantes las mejores prácticas de operación del ambiente de Google Colab/Cloud, realizando una minuciosa demostración en vivo narrando todas las consideraciones pertinentes para evitar accidentes, daños a terceros y a equipos.
Dar soporte en el acto a  l@s estudiantes que necesitan ayuda para terminar de configurar el ambiente Google Cloud .
<br>Repaso del concepto de  partición de un dataset al azar, estratificada en la clase
<br>Dado que es el primer script que se mostrará en vivo a l@s estudiantes,  realizar una visita guiada del  mismo, su estructura y detalles.
<br>Repaso del algoritmo Arbol de Decisión sus hiperparámetros, y la implementación con la librería  rpart.  Funciones  rpart::rpart  y rpart::predict
<br>Concepto de replicabilidad de los experimentos mediante las semillas de los generadores de secuencias de números pseudoaleatorios.
<br>Finalmente, el principal objetivo de este experimento es lograr que l@s estudiantes dimensionen la enorme variabilidad del error de medición de la ganancia de un árbol de decisión al realizar una partición <training, testing>,  contener la sorpresa de los estudiantes, descartar a la simple particion <training, testing> como método,  y construir en conjunto una solucion natural al problema.

## 1.2 Introduccion

![Particiohn Training/Testing](https://storage.googleapis.com/open-courses/austral2025-af91/labo1r/C2_E1_particion.jpg)

![Dos tipos de error](https://storage.googleapis.com/open-courses/austral2025-af91/labo1r/C2_E1_target.jpg)

## 1.3  Bibliografía

Demšar, J. [Statistical comparisons of classifiers over multiple data sets](https://www.jmlr.org/papers/volume7/demsar06a/demsar06a.pdf) J. Mach. Learn. Res. 7, 1–30 (2006).   
Starmer, J. [Machine Learning Fundamentals: Cross Validation](https://www.youtube.com/watch?v=fSytzGwwBVw)Machine Learning Fundamentals: Cross Validation, StatQuest with Josh Starmer youtube channel, (2018)
Hastie, T.[The elements of statistical learning: data mining, inference, and prediction](https://hastie.su.domains/Papers/ESLII.pdf) , volume 2. Springer, 2009  ( Chapter 7 Model Assessment and Selection )

## 1.4 Codigo

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,668150,35.7,1473300,78.7,1425908,76.2
Vcells,1236290,9.5,8388608,64.0,1978689,15.1


In [2]:
Sys.time()

[1] "2026-09-22 22:22:45 UTC"

* Instalacion de la libreria  rpart.plot  para dibujar el arbol
* invocacion de las librerias  **data.table** y  **rpart**

In [3]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart



In [4]:
if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: R.utils

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘R.utils’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘R.oo’, ‘R.methodsS3’


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The fol

###  Accion a Realizar
PARAM$semilla  debe tener su primer semilla aleatoria

In [40]:
PARAM <- list()
PARAM$semilla <- 810013  # aqui debe ir su primer semilla
PARAM$training_pct <- 70L  # entre  1L y 99L

PARAM$rpart <- list (
  "cp"= -1, # complejidad minima
  "minsplit"= 170, # minima cantidad de regs en un nodo para hacer el split
  "minbucket"= 70, # minima cantidad de regs en una hoja
  "maxdepth"= 7 # profundidad máxima del arbol
)


In [41]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa

# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(
    data, division, agrupa= "",
    campo= "fold", start= 1, seed= NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from= start, length.out= length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by= agrupa
  ]
}


In [42]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp0201"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [43]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")
nrow(dataset)

[1] 329555

In [44]:
dataset[, .N, list(foto_mes, clase_ternaria)]

foto_mes,clase_ternaria,N
<int>,<chr>,<int>
202107,CONTINUA,162077
202107,BAJA+2,1304
202107,BAJA+1,1098
202109,,165076


In [45]:
# trabajo solo con los datos de 202107, ultimo mes con clase_ternaria completa
# filtro datos
dataset <- dataset[foto_mes==202107]

invisible(gc(full=TRUE, verbose=FALSE)) # garbage collection

nrow(dataset)
dataset[, .N, clase_ternaria]

[1] 164479

clase_ternaria,N
<chr>,<int>
CONTINUA,162077
BAJA+2,1304
BAJA+1,1098


In [46]:
# particiono estratificadamente el dataset 70%, 30%
particionar(dataset,
  division= c(PARAM$training_pct, 100L -PARAM$training_pct),
  agrupa= "clase_ternaria",
  seed= PARAM$semilla # aqui se usa SU semilla
)

In [47]:
# genero el modelo
# quiero predecir clase_ternaria a partir del resto
# fold==1  es training,  el 70% de los datos
modelo <- rpart("clase_ternaria ~ .",
  data= dataset[fold == 1],  # fold==1  es training, el 70% de los datos
  xval= 0,
  control= PARAM$rpart # aqui van los parametros
)


In [48]:
# aplico el modelo a los datos de testing
prediccion <- predict(modelo, # el modelo que genere recien
  dataset[fold == 2], # fold==2  es testing, el 30% de los datos
  type= "prob"
) # type= "prob"  es que devuelva la probabilidad


In [49]:
tb_prediccion <- as.data.table(list(
  "clase_ternaria"= dataset[fold == 2, clase_ternaria],
  "prob"= prediccion[, "BAJA+2"]
))

In [50]:
# calculo la ganancia de cada registro
tb_prediccion[, ganancia := ifelse(clase_ternaria == "BAJA+2", 975000, -25000)]

In [51]:
# calculo la clase
tb_prediccion[, Predicted := prob > (1/40) ]

In [52]:
ganancia_test <-  tb_prediccion[ Predicted==TRUE,  sum(ganancia)]

In [53]:
# normalizo la ganancia
ganancia_test_normalizada <- ganancia_test / (( 100 - PARAM$training_pct ) / 100 )

In [54]:
estimulos <- tb_prediccion[ Predicted==TRUE, .N]
aciertos <- tb_prediccion[ Predicted & clase_ternaria == "BAJA+2", .N]


In [55]:
# Resultado Final
cat("Testing total: ", dataset[fold == 2, .N], "\n")
cat("Testing BAJA+2: ", dataset[fold == 2 & clase_ternaria == "BAJA+2", .N], "\n")

cat("Estimulos: ", estimulos, "\n")
cat("Aciertos (BAJA+2): ", aciertos, "\n")

cat("Ganancia en testing (normalizada): ", ganancia_test_normalizada, "\n")


Testing total:  49348 
Testing BAJA+2:  395 
Estimulos:  3264 
Aciertos (BAJA+2):  222 
Ganancia en testing (normalizada):  4.68e+08 


### Acciones a realizar
* Reportar la  *Ganancia en testing (normalizada)* <br> en la planilla colaborativa hoja  **C2-1sem**
* Se discutirá en clase la variabilidad de las ganancias obtenidas de distintos alumnos



---



# 2  Medición Monte Carlo Cross Validation
## Clase 02  Experimento 2

## 2.1  Objetivos
Mostrar el funcionamiento de la Montecarlo Cross Validation y que l@s estudiantes aprecien la disminución de la varianza de dicha metodología.
Relación con el Teorema Central del Límite

## 2.2 Introduccion

![Montecarlo](https://storage.googleapis.com/open-courses/austral2025-af91/labo1r/C2_E2_montecarlo.jpg)

![Curva normal](https://storage.googleapis.com/open-courses/austral2025-af91/labo1r/C2_E2_normal.jpg)

![Teorema Central del Limite](https://storage.googleapis.com/open-courses/austral2025-af91/labo1r/C2_E2_teoremacentralLimite.jpg)



## 2.3  Bibliografía

Demšar, J. [Statistical comparisons of classifiers over multiple data sets](https://www.jmlr.org/papers/volume7/demsar06a/demsar06a.pdf) J. Mach. Learn. Res. 7, 1–30 (2006).   
Starmer, J. [Machine Learning Fundamentals: Cross Validation](https://www.youtube.com/watch?v=fSytzGwwBVw)Machine Learning Fundamentals: Cross Validation, StatQuest with Josh Starmer youtube channel, (2018)
Hastie, T.[The elements of statistical learning: data mining, inference, and prediction](https://hastie.su.domains/Papers/ESLII.pdf) , volume 2. Springer, 2009  ( Chapter 7 Model Assessment and Selection )

## 2.4 Codigo  Montecarlo

El código de la Montecarlo Cross Validation es notablemente más complejo que el anterior de una simple partición training/testing
<br> Se crea la funcion ArbolEstimarGanancia()  que dada una semilla realiza la partición  training/testing, entrena en training, aplica el modelo a testing, y se calculan las ganancias
<br> Se utiliza la funcion **mcmapply**  de R para aplicar ArbolEstimarGanancia a todos los elementos del vector de semillas

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [56]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,823013,44.0,1473300,78.7,1473300,78.7
Vcells,1613232,12.4,94763342,723.0,118364237,903.1


* Instalacion de la libreria  rpart.plot  para dibujar el arbol
* invocacion de las librerias  **data.table** y  **rpart**

In [57]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")

Loading required package: parallel



In [58]:
if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

###  Accion a Realizar
PARAM$semillas  debe tener sus cinco semillas aleatorias

In [93]:
PARAM <- list()
PARAM$semillas <- c(810013, 433229, 920021, 300017, 666667)  # aqui debe ir sus CINCO
PARAM$training_pct <- 70L  # entre  1L y 99L

PARAM$rpart <- list (
  "cp" = -1, # complejidad minima
  "minsplit"= 250, # minima cantidad de regs en un nodo para hacer el split
  "minbucket"= 100, # minima cantidad de regs en una hoja
  "maxdepth"= 6 # profundidad máxima del arbol
)



In [94]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa

# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(
    data, division, agrupa = "",
    campo = "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by = agrupa
  ]
}


In [95]:

ArbolEstimarGanancia <- function(semilla, param_basicos) {
  # particiono estratificadamente el dataset
  particionar(dataset,
    division= c(param_basicos$training_pct, 100L -param_basicos$training_pct),
    agrupa= "clase_ternaria",
    seed= semilla # aqui se usa SU semilla
  )

  # genero el modelo
  # predecir clase_ternaria a partir del resto
  modelo <- rpart("clase_ternaria ~ .",
    data= dataset[fold == 1], # fold==1  es training,  el 70% de los datos
    xval= 0,
    control= param_basicos$rpart
  ) # aqui van los parametros del arbol

  # aplico el modelo a los datos de testing
  prediccion <- predict(modelo, # el modelo que genere recien
    dataset[fold == 2], # fold==2  es testing, el 30% de los datos
    type= "prob"
  ) # type= "prob"  es que devuelva la probabilidad

  # prediccion es una matriz con TRES columnas,
  #  llamadas "BAJA+1", "BAJA+2"  y "CONTINUA"
  # cada columna es el vector de probabilidades


  # calculo la ganancia en testing  qu es fold==2
  ganancia_test <- dataset[
    fold== 2,
    sum(ifelse(prediccion[, "BAJA+2"] > 0.025,
      ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
      0
    ))
  ]

  # escalo la ganancia como si fuera todo el dataset
  ganancia_test_normalizada <- ganancia_test / (( 100 - PARAM$training_pct ) / 100 )

  return(list(
    "semilla"= semilla,
    "testing"= dataset[fold == 2, .N],
    "testing_pos"= dataset[fold == 2 & clase_ternaria == "BAJA+2", .N],
    "envios"= dataset[fold == 2, sum(prediccion[, "BAJA+2"] > 0.025)],
    "aciertos"= dataset[
        fold == 2,
        sum(prediccion[, "BAJA+2"] > 0.025 & clase_ternaria == "BAJA+2")
    ],
    "ganancia_test"= ganancia_test_normalizada
  ))
}


In [96]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp0202"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [97]:
dataset <- fread("/content/datasets/dataset_pequeno.csv")
nrow(dataset)

[1] 329555

In [98]:
# trabajo solo con los datos de 202107, ultimo mes con clase_ternaria completa
# filtro datos
dataset <- dataset[foto_mes==202107]

invisible(gc(full=TRUE, verbose=FALSE)) # garbage collection

nrow(dataset)
dataset[, .N, list(foto_mes,clase_ternaria)]

[1] 164479

foto_mes,clase_ternaria,N
<int>,<chr>,<int>
202107,CONTINUA,162077
202107,BAJA+2,1304
202107,BAJA+1,1098


In [99]:

# la funcion mcmapply  llama a la funcion ArbolEstimarGanancia
#  tantas veces como valores tenga el vector  PARAM$semillas
salidas <- mcmapply(ArbolEstimarGanancia,
  PARAM$semillas, # paso el vector de semillas
  MoreArgs= list(PARAM), # aqui paso el segundo parametro
  SIMPLIFY= FALSE,
  mc.cores= detectCores()
)

# muestro la lista de las salidas en testing
#  para la particion realizada con cada semilla
salidas


[[1]]
[[1]]$semilla
[1] 810013

[[1]]$testing
[1] 49348

[[1]]$testing_pos
[1] 395

[[1]]$envios
[1] 3573

[[1]]$aciertos
[1] 231

[[1]]$ganancia_test
[1] 472250000


[[2]]
[[2]]$semilla
[1] 433229

[[2]]$testing
[1] 49343

[[2]]$testing_pos
[1] 392

[[2]]$envios
[1] 3283

[[2]]$aciertos
[1] 216

[[2]]$ganancia_test
[1] 446416667


[[3]]
[[3]]$semilla
[1] 920021

[[3]]$testing
[1] 49348

[[3]]$testing_pos
[1] 393

[[3]]$envios
[1] 3036

[[3]]$aciertos
[1] 214

[[3]]$ganancia_test
[1] 460333333


[[4]]
[[4]]$semilla
[1] 300017

[[4]]$testing
[1] 49347

[[4]]$testing_pos
[1] 391

[[4]]$envios
[1] 3073

[[4]]$aciertos
[1] 205

[[4]]$ganancia_test
[1] 427250000


[[5]]
[[5]]$semilla
[1] 666667

[[5]]$testing
[1] 49346

[[5]]$testing_pos
[1] 396

[[5]]$envios
[1] 3828

[[5]]$aciertos
[1] 248

[[5]]$ganancia_test
[1] 507666667

In [100]:
# paso la lista a vector
tb_salida <- rbindlist(salidas)
print( tb_salida)

   semilla testing testing_pos envios aciertos ganancia_test
     <num>   <int>       <int>  <int>    <int>         <num>
1:  810013   49348         395   3573      231     472250000
2:  433229   49343         392   3283      216     446416667
3:  920021   49348         393   3036      214     460333333
4:  300017   49347         391   3073      205     427250000
5:  666667   49346         396   3828      248     507666667


In [101]:
# finalmente calculo la media (promedio)  de las ganancias
cat( "ganancia promedio: ", tb_salida[, mean(ganancia_test)], "\n" )

ganancia promedio:  462783333 


### Acciones a realizar
* Reportar la  *Ganancia Promedio* <br> en la planilla colaborativa hoja  **C2-5sem**
* Se discutirá en clase la variabilidad de estos "ganancia promedio de cinco semillas"



---



# 3  Generando n semillas en forma automatica
## Clase 02  Experimento 3

## 3.1  Objetivos
Mostrar la creación de nuevas semillas a partir de la librería Primes

3.2  Código

Como generar muchas semillas a partir de una

In [68]:
# instalo y cargo la libreria  primes
if (!require("primes")) install.packages("primes")
require("primes")

Loading required package: primes

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘primes’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: primes



In [69]:
# genero numeros primos
primos <- generate_primes(min= 100000, max= 1000000)


set.seed(102191) # inicializo con mi primer semilla

# me quedo con por ejemplo 50 primos al azar
semillas <- sample(primos, 50 )

print( semillas )

 [1] 378821 964333 187049 205151 813697 344719 127217 464371 645137 480803
[11] 723319 314641 791599 937577 685649 619363 390539 465887 619657 195737
[21] 542831 438499 657539 408469 806999 491899 787181 441841 833927 745027
[31] 369007 327179 351217 819407 491083 837307 206477 439861 182899 833873
[41] 824933 641747 857083 973657 115223 635527 878197 514049 357817 278981




---



# 4  Medición  50-Monte Carlo Cross Validation
## Clase 02  Experimento 4

## 4.1  Objetivos
Calcular la ganancia del modelo utilizando 50 semillas nuevas generadas a partir de mi semilla primigenia.

## 4.4 Codigo  50 Montecarlo

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

Este punto demora en correr 60 minutos en Google Colab, con lo cual para continuar con el punto siguiente deberá abrir un nuevo Colab

limpio el ambiente de R

In [102]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,893210,47.8,1473300,78.7,1473300,78.7
Vcells,1728635,13.2,55023383,419.8,118364237,903.1


* Instalacion de la libreria  rpart.plot  para dibujar el arbol
* invocacion de las librerias  **data.table** y  **rpart**

In [103]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

if (!require("primes")) install.packages("primes")
require("primes")

###  Accion a Realizar
PARAM$semilla_primigenia  debe reemplazarse por SU primer semilla

In [109]:
PARAM <- list()
PARAM$semilla_primigenia <- 810013
PARAM$qsemillas <- 50
PARAM$training_pct <- 70L  # entre  1L y 99L

PARAM$rpart <- list (
  "cp"= -1, # complejidad minima
  "minsplit"= 170, # minima cantidad de regs en un nodo para hacer el split
  "minbucket"= 70, # minima cantidad de regs en una hoja
  "maxdepth"= 7 # profundidad máxima del arbol
)


In [110]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa

# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(
    data, division, agrupa = "",
    campo= "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by= agrupa
  ]
}


In [111]:

ArbolEstimarGanancia <- function(semilla, param_basicos) {
  # particiono estratificadamente el dataset
  particionar(dataset,
    division= c(param_basicos$training_pct, 100L -param_basicos$training_pct),
    agrupa= "clase_ternaria",
    seed= semilla # aqui se usa SU semilla
  )

  # genero el modelo
  # predecir clase_ternaria a partir del resto
  modelo <- rpart("clase_ternaria ~ .",
    data= dataset[fold == 1], # fold==1  es training,  el 70% de los datos
    xval= 0,
    control= param_basicos$rpart
  ) # aqui van los parametros del arbol

  # aplico el modelo a los datos de testing
  prediccion <- predict(modelo, # el modelo que genere recien
    dataset[fold == 2], # fold==2  es testing, el 30% de los datos
    type= "prob"
  ) # type= "prob"  es que devuelva la probabilidad

  # prediccion es una matriz con TRES columnas,
  #  llamadas "BAJA+1", "BAJA+2"  y "CONTINUA"
  # cada columna es el vector de probabilidades


  # calculo la ganancia en testing  qu es fold==2
  ganancia_test <- dataset[
    fold == 2,
    sum(ifelse(prediccion[, "BAJA+2"] > 0.025,
      ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
      0
    ))
  ]

  # escalo la ganancia como si fuera todo el dataset
  ganancia_test_normalizada <- ganancia_test / (( 100 - PARAM$training_pct ) / 100 )

  return(list(
    "semilla"= semilla,
    "testing"= dataset[fold == 2, .N],
    "testing_pos"= dataset[fold == 2 & clase_ternaria == "BAJA+2", .N],
    "envios"= dataset[fold == 2, sum(prediccion[, "BAJA+2"] > 0.025)],
    "aciertos"= dataset[
        fold == 2,
        sum(prediccion[, "BAJA+2"] > 0.025 & clase_ternaria == "BAJA+2")
    ],
    "ganancia_test"= ganancia_test_normalizada
  ))
}


In [112]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp0204"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [113]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")
nrow(dataset)

[1] 329555

In [114]:
# trabajo solo con los datos de 202107, ultimo mes con clase_ternaria completa
# filtro datos
dataset <- dataset[foto_mes==202107]

invisible(gc(full=TRUE, verbose=FALSE)) # garbage collection

nrow(dataset)
dataset[, .N, list(foto_mes,clase_ternaria)]

[1] 164479

foto_mes,clase_ternaria,N
<int>,<chr>,<int>
202107,CONTINUA,162077
202107,BAJA+2,1304
202107,BAJA+1,1098


In [115]:
# genero numeros primos
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia) # inicializo

# me quedo con PARAM$qsemillas   semillas
PARAM$semillas <- sample(primos, PARAM$qsemillas )

In [116]:

# la funcion mcmapply  llama a la funcion ArbolEstimarGanancia
#  tantas veces como valores tenga el vector  PARAM$semillas
salidas <- mcmapply(ArbolEstimarGanancia,
  PARAM$semillas, # paso el vector de semillas
  MoreArgs= list(PARAM), # aqui paso el segundo parametro
  SIMPLIFY= FALSE,
  mc.cores= detectCores()
)

# muestro la lista de las salidas en testing
#  para la particion realizada con cada semilla
salidas


[[1]]
[[1]]$semilla
[1] 470021

[[1]]$testing
[1] 49344

[[1]]$testing_pos
[1] 389

[[1]]$envios
[1] 3537

[[1]]$aciertos
[1] 249

[[1]]$ganancia_test
[1] 535250000


[[2]]
[[2]]$semilla
[1] 784081

[[2]]$testing
[1] 49340

[[2]]$testing_pos
[1] 390

[[2]]$envios
[1] 3444

[[2]]$aciertos
[1] 236

[[2]]$ganancia_test
[1] 499666667


[[3]]
[[3]]$semilla
[1] 209381

[[3]]$testing
[1] 49344

[[3]]$testing_pos
[1] 390

[[3]]$envios
[1] 3163

[[3]]$aciertos
[1] 214

[[3]]$ganancia_test
[1] 449750000


[[4]]
[[4]]$semilla
[1] 122321

[[4]]$testing
[1] 49350

[[4]]$testing_pos
[1] 398

[[4]]$envios
[1] 3439

[[4]]$aciertos
[1] 238

[[4]]$ganancia_test
[1] 506750000


[[5]]
[[5]]$semilla
[1] 609361

[[5]]$testing
[1] 49349

[[5]]$testing_pos
[1] 393

[[5]]$envios
[1] 3023

[[5]]$aciertos
[1] 210

[[5]]$ganancia_test
[1] 448083333


[[6]]
[[6]]$semilla
[1] 337969

[[6]]$testing
[1] 49345

[[6]]$testing_pos
[1] 395

[[6]]$envios
[1] 3508

[[6]]$aciertos
[1] 229

[[6]]$ganancia_test
[1] 4.71e+08


[[7]]
[[7]]$semilla
[1] 245501

[[7]]$testing
[1] 49341

[[7]]$testing_pos
[1] 391

[[7]]$envios
[1] 3728

[[7]]$aciertos
[1] 245

[[7]]$ganancia_test
[1] 5.06e+08


[[8]]
[[8]]$semilla
[1] 112031

[[8]]$testing
[1] 49343

[[8]]$testing_pos
[1] 387

[[8]]$envios
[1] 3347

[[8]]$aciertos
[1] 213

[[8]]$ganancia_test
[1] 431083333


[[9]]
[[9]]$semilla
[1] 272659

[[9]]$testing
[1] 49344

[[9]]$testing_pos
[1] 396

[[9]]$envios
[1] 3668

[[9]]$aciertos
[1] 254

[[9]]$ganancia_test
[1] 5.41e+08


[[10]]
[[10]]$semilla
[1] 165203

[[10]]$testing
[1] 49340

[[10]]$testing_pos
[1] 392

[[10]]$envios
[1] 3400

[[10]]$aciertos
[1] 231

[[10]]$ganancia_test
[1] 486666667


[[11]]
[[11]]$semilla
[1] 371971

[[11]]$testing
[1] 49340

[[11]]$testing_pos
[1] 390

[[11]]$envios
[1] 3631

[[11]]$aciertos
[1] 236

[[11]]$ganancia_test
[1] 484083333


[[12]]
[[12]]$semilla
[1] 481447

[[12]]$testing
[1] 49338

[[12]]$testing_pos
[1] 390

[[12]]$envios
[1] 3632

[[12]]$aciertos
[1] 234

[[12]]$ganancia_test
[1] 477333333


[[13]]
[[13]]$semilla
[1] 139589

[[13]]$testing
[1] 49339

[[13]]$testing_pos
[1] 386

[[13]]$envios
[1] 3557

[[13]]$aciertos
[1] 225

[[13]]$ganancia_test
[1] 453583333


[[14]]
[[14]]$semilla
[1] 626581

[[14]]$testing
[1] 49339

[[14]]$testing_pos
[1] 387

[[14]]$envios
[1] 3573

[[14]]$aciertos
[1] 229

[[14]]$ganancia_test
[1] 465583333


[[15]]
[[15]]$semilla
[1] 530293

[[15]]$testing
[1] 49335

[[15]]$testing_pos
[1] 382

[[15]]$envios
[1] 3341

[[15]]$aciertos
[1] 216

[[15]]$ganancia_test
[1] 441583333


[[16]]
[[16]]$semilla
[1] 855527

[[16]]$testing
[1] 49342

[[16]]$testing_pos
[1] 391

[[16]]$envios
[1] 3063

[[16]]$aciertos
[1] 220

[[16]]$ganancia_test
[1] 478083333


[[17]]
[[17]]$semilla
[1] 891047

[[17]]$testing
[1] 49337

[[17]]$testing_pos
[1] 390

[[17]]$envios
[1] 3391

[[17]]$aciertos
[1] 225

[[17]]$ganancia_test
[1] 467416667


[[18]]
[[18]]$semilla
[1] 661093

[[18]]$testing
[1] 49342

[[18]]$testing_pos
[1] 388

[[18]]$envios
[1] 3620

[[18]]$aciertos
[1] 222

[[18]]$ganancia_test
[1] 438333333


[[19]]
[[19]]$semilla
[1] 501827

[[19]]$testing
[1] 49348

[[19]]$testing_pos
[1] 397

[[19]]$envios
[1] 2857

[[19]]$aciertos
[1] 200

[[19]]$ganancia_test
[1] 428583333


[[20]]
[[20]]$semilla
[1] 931163

[[20]]$testing
[1] 49343

[[20]]$testing_pos
[1] 389

[[20]]$envios
[1] 2793

[[20]]$aciertos
[1] 204

[[20]]$ganancia_test
[1] 447250000


[[21]]
[[21]]$semilla
[1] 365749

[[21]]$testing
[1] 49342

[[21]]$testing_pos
[1] 388

[[21]]$envios
[1] 3360

[[21]]$aciertos
[1] 232

[[21]]$ganancia_test
[1] 493333333


[[22]]
[[22]]$semilla
[1] 358213

[[22]]$testing
[1] 49347

[[22]]$testing_pos
[1] 394

[[22]]$envios
[1] 2919

[[22]]$aciertos
[1] 224

[[22]]$ganancia_test
[1] 503416667


[[23]]
[[23]]$semilla
[1] 619741

[[23]]$testing
[1] 49337

[[23]]$testing_pos
[1] 387

[[23]]$envios
[1] 3471

[[23]]$aciertos
[1] 225

[[23]]$ganancia_test
[1] 460750000


[[24]]
[[24]]$semilla
[1] 974737

[[24]]$testing
[1] 49346

[[24

In [117]:
# paso la lista a vector
tb_salida <- rbindlist(salidas)
print( tb_salida)

    semilla testing testing_pos envios aciertos ganancia_test
      <int>   <int>       <int>  <int>    <int>         <num>
 1:  470021   49344         389   3537      249     535250000
 2:  784081   49340         390   3444      236     499666667
 3:  209381   49344         390   3163      214     449750000
 4:  122321   49350         398   3439      238     506750000
 5:  609361   49349         393   3023      210     448083333
 6:  337969   49345         395   3508      229     471000000
 7:  245501   49341         391   3728      245     506000000
 8:  112031   49343         387   3347      213     431083333
 9:  272659   49344         396   3668      254     541000000
10:  165203   49340         392   3400      231     486666667
11:  371971   49340         390   3631      236     484083333
12:  481447   49338         390   3632      234     477333333
13:  139589   49339         386   3557      225     453583333
14:  626581   49339         387   3573      229     465583333
15:  530

In [118]:
# calculo la salida
for( i in seq(10, 50, 10) )
{
  cat( i, "\t", tb_salida[ 1:i, mean(ganancia_test)], "https://drive.google.com/drive/folders/1k2IStdNT_P7MU4vIT3scUljCQTRa3eef?usp=drive_link" )
}


10 	 487525000 https://drive.google.com/drive/folders/1k2IStdNT_P7MU4vIT3scUljCQTRa3eef?usp=drive_link20 	 472854167 https://drive.google.com/drive/folders/1k2IStdNT_P7MU4vIT3scUljCQTRa3eef?usp=drive_link30 	 474291667 https://drive.google.com/drive/folders/1k2IStdNT_P7MU4vIT3scUljCQTRa3eef?usp=drive_link40 	 470952083 https://drive.google.com/drive/folders/1k2IStdNT_P7MU4vIT3scUljCQTRa3eef?usp=drive_link50 	 467761667 https://drive.google.com/drive/folders/1k2IStdNT_P7MU4vIT3scUljCQTRa3eef?usp=drive_link

In [119]:
tb_salida <- rbindlist(salidas)
g <- tb_salida$ganancia_test
mean(g[1:10])
mean(g[1:20])
mean(g[1:30])
mean(g[1:40])
mean(g[1:50])

[1] 487525000

[1] 472854167

[1] 474291667

[1] 470952083

[1] 467761667

### Acciones a realizar
* Reportar la  los resultados en la planilla colaborativa hoja  **C2-Nsem**
* Se discutirá en clase la variabilidad de las distintas cantidades de semillas



---





---



# 5 Comparando dos distintos  modelos
## Clase 02  Experimento 5

## 5.1  Objetivos
Presentar a los alumnos la dificultad que se presenta ante la comparación de dos modelos, uno posee un poder predictivo claramente mayor al otro.
Comparación simple de media de ganancias, versus probabilidad que un modelo sea superior a otro.

## 5.2 Introduccion
Estos son los dos arboles de muy distinta profundidad.
<br>¿Cuál de ellos es mejor? ¿Cuál elijo?

| Hiperparámetro | Arbol 1 | Arbol 2 |
| --- | --: |  ---: |
| cp | -1 | -1 |
|minsplit | 170 | 250 |
|minbucket | 70 | 125 |
|maxdepth | 7 | 20 |



## 5.3 Codigo dos distintos modelos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

El tiempo de corrida es de alrededor de 40 minutos

limpio el ambiente de R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

* Instalacion de la libreria  rpart.plot  para dibujar el arbol
* invocacion de las librerias  **data.table** y  **rpart**

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")

if (!require("primes")) install.packages("primes")
require("primes")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

require("ggplot2")

###  Accion a Realizar
PARAM$semilla_primigenia  debe reemplazarse por SU primer semilla

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191
PARAM$qsemillas <- 20
PARAM$training_pct <- 70L     # entre 1 y 100


In [ ]:
# los dos arboles
PARAM$rpart1 <- list (
  "cp" = -1,
  "minsplit" = 170,
  "minbucket" = 70,
  "maxdepth" = 7
)


PARAM$rpart2 <- list (
  "cp" = -1,
  "minsplit" = 250,
  "minbucket" = 125,
  "maxdepth" = 20
)

In [ ]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa

# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(
    data, division, agrupa = "",
    campo = "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by = agrupa
  ]
}


In [ ]:
DosArbolesEstimarGanancia <- function(semilla, training_pct, param_rpart1, param_rpart2) {

  # fuerzo impresion
  print( paste( semilla, Sys.time()) )
  flush.console()

  # particiono estratificadamente el dataset
  particionar(dataset,
    division= c(training_pct, 100L -training_pct),
    agrupa= "clase_ternaria",
    seed= semilla # aqui se usa SU semilla
  )

  # genero el modelo
  # predecir clase_ternaria a partir del resto
  modelo1 <- rpart("clase_ternaria ~ .",
    data= dataset[fold == 1], # fold==1  es training,  el 70% de los datos
    xval= 0,
    control= param_rpart1
  ) # aqui van los parametros del arbol

  # aplico el modelo a los datos de testing
  prediccion1 <- predict(modelo1, # el modelo que genere recien
    dataset[fold == 2], # fold==2  es testing, el 30% de los datos
    type= "prob"
  ) # type= "prob"  es que devuelva la probabilidad


  # calculo la ganancia en testing  qu es fold==2
  ganancia_test1 <- dataset[
    fold == 2,
    sum(ifelse(prediccion1[, "BAJA+2"] > 0.025,
      ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
      0
    ))
  ]

  # escalo la ganancia como si fuera todo el dataset
  ganancia_test_normalizada1 <- ganancia_test1 / (( 100 - training_pct ) / 100 )

  modelo2 <- rpart("clase_ternaria ~ .",
    data= dataset[fold == 1], # fold==1  es training,  el 70% de los datos
    xval= 0,
    control= param_rpart2
  ) # aqui van los parametros del arbol

  # aplico el modelo a los datos de testing
  prediccion2 <- predict(modelo2, # el modelo que genere recien
    dataset[fold == 2], # fold==2  es testing, el 30% de los datos
    type= "prob"
  ) # type= "prob"  es que devuelva la probabilidad


  # calculo la ganancia en testing  qu es fold==2
  ganancia_test2 <- dataset[
    fold == 2,
    sum(ifelse(prediccion2[, "BAJA+2"] > 0.025,
      ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
      0
    ))
  ]

  # escalo la ganancia como si fuera todo el dataset
  ganancia_test_normalizada2 <- ganancia_test2 / (( 100 - training_pct ) / 100 )

  return(list(
    "semilla"= semilla,
    "ganancia1"= ganancia_test_normalizada1,
    "ganancia2"= ganancia_test_normalizada2
  ))
}

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp0205"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")
nrow(dataset)

In [ ]:
# trabajo solo con los datos de 202107, ultimo mes con clase_ternaria completa
# filtro datos
dataset <- dataset[foto_mes==202107]

invisible(gc(full=TRUE, verbose=FALSE)) # garbage collection

nrow(dataset)
dataset[, .N, list(foto_mes,clase_ternaria)]

In [ ]:
detectCores()

In [ ]:
# genero numeros primos
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia) # inicializo

# me quedo con PARAM$qsemillas   semillas
PARAM$semillas <- sample(primos, PARAM$qsemillas )
PARAM$semillas

In [ ]:
# demora interminables 60 minutos en correr en Google Colab
Sys.time()

salidas <- mcmapply( DosArbolesEstimarGanancia,
  PARAM$semillas, # paso el vector de semillas
  MoreArgs= list(PARAM$training_pct, PARAM$rpart1, PARAM$rpart2), # aqui paso el segundo parametro
  SIMPLIFY= FALSE,
  mc.cores= detectCores()
)

Sys.time()

In [ ]:
# paso la lista a vector
tb_salida <- rbindlist(salidas)
tb_salida

In [ ]:
# creo tabla nueva con ambas ganancias en un solo campo
tb_unificado <- melt(tb_salida,
  measure= c("ganancia1", "ganancia2")
)

tb_unificado

In [ ]:
gra_boxplot <- ggplot( tb_unificado, aes(x=variable, y=value, fill=variable) )+
  geom_boxplot() +
  scale_fill_manual(values = alpha( c("green", "purple"), 0.10)) +
  labs(x = "Arbol", y = "Ganancia") +
  theme_minimal()

print(gra_boxplot)

In [ ]:
# imprimo BoxPlot a .pdf
pdf("boxplot_dos.pdf")
print(gra_boxplot)
dev.off()

In [ ]:
gra_densidad <- ggplot( tb_salida, aes(x=ganancia1)) + geom_density(alpha=0.25, fill="green", color="green")  +
  geom_density(data=tb_salida, aes(x=ganancia2), fill="purple", color="purple",  alpha=0.10)

print(gra_densidad)

In [ ]:
# imprimo en un .pdf  que va a la carpeta del experimento
pdf("densidad_dos.pdf")
print(gra_densidad)
dev.off()

In [ ]:
print( tb_salida[ , list( "arbol1" = mean( ganancia1),  "arbol2" = mean(ganancia2) ) ] )

print( tb_salida[ , list( "prob( m1 > m2)" = sum(ganancia1 > ganancia2 )/ .N ) ]  )

### Acciones a realizar
* Reportar la  los resultados en la planilla colaborativa hoja  **C2-dosModelos**
* ¿Todos los alumnos hubieran elegido el mismo modelo aun con 20 semillas?



---



# 6 Comparando dos buenos  modelos
## Clase 02  Experimento 6

## 6.1  Objetivos
Presentar a los alumnos la dificultad que surge en la comparación de dos muy distintos, pero ambos buenos, modelos predictivos, y el costo computacional asociado a esa comparación
<br>Comparación simple de media de ganancias, versus probabilidad que un modelo sea superior a otro.
<br>Limitaciones de la MonteCarlo Cross Validation
<br>Existencia del Test de Wilcoxon

## 6.2 Introduccion
Estos son los dos arboles de muy distinta profundidad.
<br>¿Cuál de ellos es mejor? ¿Cuál elijo?

| Hiperparámetro | Arbol 1 | Arbol 2 |
| --- | --: |  ---: |
| cp | -1 | -1 |
|minsplit | 700 | 115 |
|minbucket | 350 | 5 |
|maxdepth | 8 | 6 |



## 6.3 Dos buenos modelos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

El tiempo de corrida es de alrededor de 50 minutos

limpio el ambiente de R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

* Instalacion de la libreria  rpart.plot  para dibujar el arbol
* invocacion de las librerias  **data.table** y  **rpart**

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

if (!require("primes")) install.packages("primes")
require("primes")

require("ggplot2")

###  Accion a Realizar
PARAM$semilla_primigenia  debe reemplazarse por SU primer semilla

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191
PARAM$qsemillas <- 50
PARAM$training_pct <- 70L     # entre 1 y 100


In [ ]:
# los dos arboles
PARAM$rpart1 <- list (
  "cp" = -1,
  "minsplit" = 750,
  "minbucket" = 350,
  "maxdepth" = 8
)


PARAM$rpart2 <- list (
  "cp" = -1,
  "minsplit" = 115,
  "minbucket" = 5,
  "maxdepth" = 6
)


In [ ]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa

# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(
    data, division, agrupa = "",
    campo = "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by = agrupa
  ]
}


In [ ]:
DosArbolesEstimarGanancia <- function(semilla, training_pct, param_rpart1, param_rpart2) {
  # fuerzo impresion
  print( paste( semilla, Sys.time()) )
  flush.console()

  # particiono estratificadamente el dataset
  particionar(dataset,
    division = c(training_pct, 100L -training_pct),
    agrupa = "clase_ternaria",
    seed = semilla # aqui se usa SU semilla
  )

  # genero el modelo
  # predecir clase_ternaria a partir del resto
  modelo1 <- rpart("clase_ternaria ~ .",
    data = dataset[fold == 1], # fold==1  es training,  el 70% de los datos
    xval = 0,
    control = param_rpart1
  ) # aqui van los parametros del arbol

  # aplico el modelo a los datos de testing
  prediccion1 <- predict(modelo1, # el modelo que genere recien
    dataset[fold == 2], # fold==2  es testing, el 30% de los datos
    type = "prob"
  ) # type= "prob"  es que devuelva la probabilidad


  # calculo la ganancia en testing  qu es fold==2
  ganancia_test1 <- dataset[
    fold == 2,
    sum(ifelse(prediccion1[, "BAJA+2"] > 0.025,
      ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
      0
    ))
  ]

  # escalo la ganancia como si fuera todo el dataset
  ganancia_test_normalizada1 <- ganancia_test1 / (( 100 - training_pct ) / 100 )

  modelo2 <- rpart("clase_ternaria ~ .",
    data= dataset[fold == 1], # fold==1  es training,  el 70% de los datos
    xval= 0,
    control= param_rpart2
  ) # aqui van los parametros del arbol

  # aplico el modelo a los datos de testing
  prediccion2 <- predict(modelo2, # el modelo que genere recien
    dataset[fold == 2], # fold==2  es testing, el 30% de los datos
    type= "prob"
  ) # type= "prob"  es que devuelva la probabilidad


  # calculo la ganancia en testing  qu es fold==2
  ganancia_test2 <- dataset[
    fold == 2,
    sum(ifelse(prediccion2[, "BAJA+2"] > 0.025,
      ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
      0
    ))
  ]

  # escalo la ganancia como si fuera todo el dataset
  ganancia_test_normalizada2 <- ganancia_test2 / (( 100 - training_pct ) / 100 )

  return(list(
    "semilla"= semilla,
    "ganancia1"= ganancia_test_normalizada1,
    "ganancia2"= ganancia_test_normalizada2
  ))
}

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp0206"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")
nrow(dataset)

In [ ]:
# trabajo solo con los datos de 202107, ultimo mes con clase_ternaria completa
# filtro datos
dataset <- dataset[foto_mes==202107]

invisible(gc(full=TRUE, verbose=FALSE)) # garbage collection

nrow(dataset)
dataset[, .N, list(foto_mes,clase_ternaria)]

In [ ]:
# genero numeros primos
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia) # inicializo

# me quedo con PARAM$qsemillas   semillas
PARAM$semillas <- sample(primos, PARAM$qsemillas )

In [ ]:
# la funcion mcmapply  llama a la funcion DosArbolesEstimarGanancia
#  tantas veces como valores tenga el vector  PARAM$semillas
Sys.time()

salidas <- mcmapply( DosArbolesEstimarGanancia,
  PARAM$semillas, # paso el vector de semillas
  MoreArgs= list(PARAM$training_pct, PARAM$rpart1, PARAM$rpart2), # aqui paso el segundo parametro
  SIMPLIFY= FALSE,
  mc.cores= detectCores()
)

Sys.time()

In [ ]:
# paso la lista a vector
tb_salida <- rbindlist(salidas)
tb_salida

In [ ]:
grafico <- ggplot( tb_salida, aes(x=ganancia1), fill="green", color="green") + geom_density(alpha=0.25)  +
             geom_density(data=tb_salida, aes(x=ganancia2), fill="purple", color="purple",  alpha=0.10)

print(grafico)

In [ ]:
# imprimo en un .pdf  que va a la carpeta del experimento
pdf("densidad_dos.pdf")
print(grafico)
dev.off()

In [ ]:
# medias de las ganancias
print( tb_salida[ , list( "arbol1" = mean( ganancia1),  "arbol2" = mean(ganancia2) ) ] )


In [ ]:
# probabilidad que m1 sea mayor a m2
print( tb_salida[ , list( "prob( m1 > m2)" = sum(ganancia1 > ganancia2 )/ .N ) ]  )

### Acciones a realizar
* Reportar la  los resultados en la planilla colaborativa hoja  **C2-dosBuenos**




---



# 7 Test de Wilcoxon
## Clase 02  Experimento 7

## 7.1  Objetivos

¿Como calcular la cantidad mínima de semillas que hacen falta para tener cierta certeza que un modelo es superior a otro?

## 7.2  Introduccion

| Hiperparámetro | Arbol 1 | Arbol 2 |
| --- | --: |  ---: |
| cp | -1 | -1 |
|minsplit | 170 | 250 |
|minbucket | 70 | 125 |
|maxdepth | 7 | 20 |

<br>
<br>



## 7.3 Codigo Test de Wilcoxon

In [ ]:
# 1 sola ganancia
wilcox.test(
  tb_salida[1:1, ganancia1],
  tb_salida[1:1, ganancia2],
  paired = TRUE
)


In [ ]:
# 2 ganancias
wilcox.test(
  tb_salida[1:2, ganancia1],
  tb_salida[1:2, ganancia2],
  paired= TRUE
)


In [ ]:
for( i in 1:50)
{
  w <- wilcox.test(
     tb_salida[1:i, ganancia1],
     tb_salida[1:i, ganancia2],
     paired= TRUE
    )
  cat( i, w$p.value, "\n")
}



---



# 8 Comparando automaticamente modelos con test de Wilcoxon
## Clase 02  Experimento 8

## 8.1  Objetivos

Proveer un script que permita comparar dos arboles de decisión con la librería rpart utilizando la menor cantidad de cómputo que permita el Test de Wilxcoxon

## 8.2  Introduccion

| Hiperparámetro | Arbol 1 | Arbol 2 |
| --- | --- |  --- |
| cp | -1 | -1 |
|minsplit | 1050 | 650 |
|minbucket | 550 | 300 |
|maxdepth |67 | 6 |




## 8.3 Codigo comparacion automatica Wilcoxon

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

* Instalacion de la libreria  rpart.plot  para dibujar el arbol
* invocacion de las librerias  **data.table** y  **rpart**

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

if (!require("primes")) install.packages("primes")
require("primes")

require("ggplot2")

###  Accion a Realizar
PARAM$semilla_primigenia  debe reemplazarse por SU primer semilla

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191
PARAM$qsemillas_tope <- 50
PARAM$training_pct <- 70L     # entre 1 y 100


In [ ]:
# los dos arboles
PARAM$rpart1 <- list (
  "cp" = -1,
  "minsplit" = 800,
  "minbucket" = 400,
  "maxdepth" = 7
)


PARAM$rpart2 <- list (
  "cp" = -1,
  "minsplit" = 650,
  "minbucket" = 300,
  "maxdepth" = 6
)


In [ ]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa

# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(
    data, division, agrupa= "",
    campo= "fold", start= 1, seed= NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from= start, length.out= length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by= agrupa
  ]
}


In [ ]:
DosArbolesEstimarGanancia <- function(semilla, training_pct, param_rpart1, param_rpart2) {
  # particiono estratificadamente el dataset
  particionar(dataset,
    division= c(training_pct, 100L -training_pct),
    agrupa= "clase_ternaria",
    seed= semilla # aqui se usa SU semilla
  )

  # genero el modelo
  # predecir clase_ternaria a partir del resto
  modelo1 <- rpart("clase_ternaria ~ .",
    data= dataset[fold == 1], # fold==1  es training,  el 70% de los datos
    xval= 0,
    control= param_rpart1
  ) # aqui van los parametros del arbol

  # aplico el modelo a los datos de testing
  prediccion1 <- predict(modelo1, # el modelo que genere recien
    dataset[fold == 2], # fold==2  es testing, el 30% de los datos
    type= "prob"
  ) # type= "prob"  es que devuelva la probabilidad


  # calculo la ganancia en testing  qu es fold==2
  ganancia_test1 <- dataset[
    fold == 2,
    sum(ifelse(prediccion1[, "BAJA+2"] > 0.025,
      ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
      0
    ))
  ]

  # escalo la ganancia como si fuera todo el dataset
  ganancia_test_normalizada1 <- ganancia_test1 / (( 100 - training_pct ) / 100 )

  modelo2 <- rpart("clase_ternaria ~ .",
    data= dataset[fold == 1], # fold==1  es training,  el 70% de los datos
    xval= 0,
    control= param_rpart2
  ) # aqui van los parametros del arbol

  # aplico el modelo a los datos de testing
  prediccion2 <- predict(modelo2, # el modelo que genere recien
    dataset[fold == 2], # fold==2  es testing, el 30% de los datos
    type= "prob"
  ) # type= "prob"  es que devuelva la probabilidad


  # calculo la ganancia en testing  qu es fold==2
  ganancia_test2 <- dataset[
    fold == 2,
    sum(ifelse(prediccion2[, "BAJA+2"] > 0.025,
      ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
      0
    ))
  ]

  # escalo la ganancia como si fuera todo el dataset
  ganancia_test_normalizada2 <- ganancia_test2 / (( 100 - training_pct ) / 100 )

  return(list(
    "semilla"= semilla,
    "ganancia1"= ganancia_test_normalizada1,
    "ganancia2"= ganancia_test_normalizada2
  ))
}

In [ ]:
# 1  ->  el modelo 1 es mejor
# 2  ->  el modelo 2 es mejor
# 0  ->  No se pudo determinar con el tope de qsemillas_tope


MejorArbol <- function( qsemillas_tope, training_pct, param_rpart1, param_rpart2) {

  # genero numeros primos
  primos <- generate_primes(min= 100000, max= 1000000)
  set.seed(PARAM$semilla_primigenia) # inicializo
  # me quedo con PARAM$qsemillas   semillas
  semillas <- sample(primos, qsemillas_tope )

  pvalue <- 1.0
  isem <- 1
  vgan1 <- c() # almaceno ganancias del modelo1
  vgan2 <- c() # almaceno ganancias del modelo2

  while( (isem <= qsemillas_tope)  & (pvalue > 0.05) ) {

    res <- DosArbolesEstimarGanancia(
       semillas[ isem ],
       training_pct,
       param_rpart1,
       param_rpart2
    )

    vgan1 <- c( vgan1, res$ganancia1 )
    vgan2 <- c( vgan2, res$ganancia2 )

    wt <- wilcox.test( vgan1, vgan2, paired=TRUE )
    pvalue <- wt$p.value

    cat( isem, res$ganancia1, res$ganancia2, pvalue, "\n" )
    flush.console()
    isem <- isem + 1
  }

  out <- 0

  if( pvalue < 0.05 & mean(vgan1) > mean(vgan2)  )  out <- 1
  if( pvalue < 0.05 & mean(vgan1) < mean(vgan2)  )  out <- 2


  return( list( "out"= out,
    "qsemillas"= length(vgan1),
    "m1"= mean( vgan1 ),
    "m2"= mean( vgan2 )
   ) )
}


In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp0208"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")
nrow(dataset)

In [ ]:
# trabajo solo con los datos de 202107, ultimo mes con clase_ternaria completa
# filtro datos
dataset <- dataset[foto_mes==202107]

invisible(gc(full=TRUE, verbose=FALSE)) # garbage collection

nrow(dataset)
dataset[, .N, list(foto_mes,clase_ternaria)]

In [ ]:
Sys.time()

comparacion <- MejorArbol(
   PARAM$qsemillas_tope,
   PARAM$training_pct,
   PARAM$rpart1,
   PARAM$rpart2
 )


print( comparacion )

Sys.time()

### Acciones a realizar
* Reportar la  los resultados en la planilla colaborativa hoja  **C2-Wilcox**




---

